# Qwen2.5-1.5B + LoRA: `[TRACK]` tracker

Runs [`training/qwen_track.py`](qwen_track.py): trains the tracker's LoRA adapter on the MultiWOZ +
synthetic `[TRACK]` rows, evaluates it per source (joint / slot / per-slot accuracy, slot F1, invalid
JSON), and pushes it to `makan-where-cs425/makan-qwen-track`, which the backend serves as
`MAKAN_TRACKER_ADAPTER`. All training code is in the script; this notebook only sets up and calls it.

- **Colab:** GPU runtime (T4 is enough), secrets `HF_TOKEN` (write access, to push) and `GITHUB_TOKEN`
  (read access, to clone the repo), each with notebook access. Checkpoints go to Drive, so after a
  disconnect, run all again and training resumes from the newest one.
- **GPU cluster / any Jupyter:** start Jupyter from `training/` in a checkout, with
  `backend/requirements-ml.txt` + `bitsandbytes` installed and `HF_TOKEN` exported.

In [ ]:
QUICK_TRIAL = False   # True: 200 rows, 1 epoch, 20 eval rows - check the setup in minutes before the real run
EXTRA_ARGS = ""       # more qwen_track.py options, e.g. "--no-4bit" (plain LoRA, faster) or "--epochs 3"
HUB_REPO = "makan-where-cs425/makan-qwen-track"
REPO_URL = "github.com/deniseLie/CS425-Makan-Where.git"

In [ ]:
import os, subprocess, sys
from pathlib import Path

try:
    from google.colab import drive, userdata
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB:
    drive.mount("/content/drive")
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    REPO = Path("/content/CS425-Makan-Where")
    OUT = Path("/content/drive/MyDrive/makan-qwen/track" + ("_trial" if QUICK_TRIAL else ""))
    if not REPO.exists():
        url = f"https://{userdata.get('GITHUB_TOKEN')}@{REPO_URL}"
        # no check=True: its error message would print the URL, token included
        if subprocess.run(["git", "clone", "-q", "--depth", "1", url, str(REPO)]).returncode:
            raise RuntimeError("git clone failed: check the GITHUB_TOKEN secret")
        subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", f"https://{REPO_URL}"])
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "peft", "accelerate", "bitsandbytes"], check=True)
else:
    REPO = Path.cwd().parent              # run from training/ in a local checkout
    OUT = REPO / "outputs" / ("qwen_track_trial" if QUICK_TRIAL else "qwen_track")   # outputs/ is gitignored

SCRIPT = REPO / "training" / "qwen_track.py"
FINAL = OUT / "final"   # the trained adapter + tokenizer
assert SCRIPT.exists(), f"{SCRIPT} missing: is qwen_track.py pushed to GitHub main?"
ARGS = f"--out '{OUT}' " + ("--limit 200 --epochs 1 --eval-n 20 " if QUICK_TRIAL else "") + EXTRA_ARGS
print("script:", SCRIPT, "| out:", OUT, "| args:", ARGS)

## Train and evaluate on dev

Prints the training log (dev loss about every 10% of the run), then the dev scores for MultiWOZ and
synthetic separately. Predictions go to `OUT/eval_track_dev_<source>.jsonl`; the adapter to `OUT/final`.

In [ ]:
!python "{SCRIPT}" {ARGS}

## Test scores (for the report)

The full MultiWOZ test set (the synthetic data has no test split). Run this once, on the final adapter.

In [ ]:
!python "{SCRIPT}" --eval-only "{FINAL}" --split test --eval-n 0 --out "{OUT}"

## Push to the Hub

Only once the dev scores look good. Uploads `OUT/final` (adapter + tokenizer) to the private repo; the
Hub keeps earlier versions in its history. Then serve it with `MAKAN_TRACKER=hf` in the backend's `.env`.

In [ ]:
from huggingface_hub import HfApi

assert not QUICK_TRIAL, "that was a trial run: set QUICK_TRIAL = False and train for real first"
api = HfApi()
api.create_repo(HUB_REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=str(FINAL), repo_id=HUB_REPO, commit_message="[TRACK] adapter")
print(f"pushed https://huggingface.co/{HUB_REPO}")